# Suppress SAE features during generation

This notebook launches the flower feature-suppression experiment from the companion `debiasing` checkout. The script selects candidate units from its training split, attenuates their decoder contributions during SDXL Turbo generation, and compares positive and negative prompts with a random-feature control. It saves comparison grids and metrics.

Run on a CUDA GPU node after the C2C targets, residuals, fine-tuned checkpoint, and box head are available. This is the current exploratory Turbo workflow; it does not use the SDXL Base SAE.

In [ ]:
from pathlib import Path
import sys
import subprocess
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as DisplayImage

NOTEBOOK_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                     if p.name == "sae-demo-notebooks" and (p / "sae-feature-suppression").exists())
DEBIASING_ROOT = NOTEBOOK_ROOT.parent  # Edit this if debiasing is elsewhere.
TRAINING_DIR = DEBIASING_ROOT / "sae-training"
RUNNER = DEBIASING_ROOT / "flower_sae_intervention/ablate_flower_features.py"
TARGETS = TRAINING_DIR / "c2c_box_targets.csv"
RESIDUALS = TRAINING_DIR / "residuals"
SAE_CHECKPOINT = TRAINING_DIR / "runs/c2c_box_sae_003/sae"
BOX_HEAD = TRAINING_DIR / "runs/c2c_box_sae_003/box_head.pth"


## Configure the comparison

Strength is the fraction of the selected SAE feature contribution retained: `0` removes it, `0.5` halves it, and `1` is a no-op. The runner also creates an activity-matched random-feature control. The prompt cohort is currently fixed to East Asian woman prompts in the script.

In [ ]:
feature_count = widgets.IntSlider(value=3, min=1, max=10, step=1, description="Units:")
positive_limit = widgets.IntSlider(value=4, min=1, max=20, step=1, description="Positive:")
negative_limit = widgets.IntSlider(value=4, min=1, max=20, step=1, description="Negative:")
strengths = widgets.Text(value="0 0.5 1", description="Retain:")
button = widgets.Button(description="Run suppression test", button_style="warning")
output = widgets.Output()

def launch_suppression(_):
    with output:
        clear_output(wait=True)
        needed = [RUNNER, TARGETS, SAE_CHECKPOINT / "config.json", BOX_HEAD]
        missing = [str(p) for p in needed if not p.exists()]
        if missing or not RESIDUALS.exists():
            print("Missing companion experiment files:")
            print("\n".join(missing + ([] if RESIDUALS.exists() else [str(RESIDUALS)])))
            return
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        run_dir = TRAINING_DIR / "runs" / f"notebook_suppression_{stamp}"
        command = [sys.executable, str(RUNNER), "--targets", str(TARGETS),
                   "--residuals", str(RESIDUALS), "--sae-checkpoint", str(SAE_CHECKPOINT),
                   "--head", str(BOX_HEAD), "--output", str(run_dir),
                   "--features", str(feature_count.value),
                   "--positive-limit", str(positive_limit.value),
                   "--negative-limit", str(negative_limit.value), "--strengths"]
        command.extend(strengths.value.split())
        print("Running:", " ".join(command), flush=True)
        result = subprocess.run(command, cwd=DEBIASING_ROOT, check=False)
        print(f"Exit code: {result.returncode}")
        if result.returncode == 0:
            print(f"Saved comparison images and metrics to {run_dir}")
            for page in sorted(run_dir.glob("all_test_comparisons_page_*.jpg")):
                display(DisplayImage(filename=str(page)))

button.on_click(launch_suppression)
display(feature_count, positive_limit, negative_limit, strengths, button, output)


Interpret the images directly. Pixel MAE measures image change, not whether a flower disappeared. Compare positives and negatives, the random control, and preservation of the person and surrounding scene. The C2C boxes are automatic labels and have not been manually verified.